In [5]:
sentences = [
    "The princess is a beautiful woman",
    "The royal family is the king and queen and their children",
    "Prince is only a boy now",
    "A boy will be a man",
    "A girl grows up to be a woman",
    "The woman is a mother to the daughter",
    "Every woman was once a girl",
    "The young girl aspires to be a wise woman",
    "The daughter has a close bond with her mother",
    "In every woman is the heart of a girl",
    "The mother love for her daughter is strong",
    "A daughter may inherit her mother traits",
    "The girl and her mother share a special relationship",
    "Womanhood is the journey from a girl to a mature individual",
    "A daughter first role model is often her mother",
    "The girl looked up to her mother, aspiring to be a strong woman",
]

In [6]:
import re


# Text Preprocessing Function
def clean_text(
        string, punctuations=r'''!()-[]{};:'"\,<>./?@#$%^&*_~''', stop_words=['the', 'a', 'and', 'is', 'be', 'will']
):
    string = re.sub(r'https?://\S+|www\.\S+', '', string)
    string = re.sub(r'<.*?>', '', string)
    for x in string.lower():
        if x in punctuations:
            string = string.replace(x, "")
    string = string.lower()
    string = ' '.join([word for word in string.split() if word not in stop_words])
    string = re.sub(r'\s+', ' ', string).strip()
    return string

In [7]:
# Creating Context Pairs
def create_context_pairs(texts, window=2):
    word_lists = []
    all_text = []

    for text in texts:
        text = clean_text(text)
        words = text.split()
        all_text += words

        for i, word in enumerate(words):
            for w in range(-window, window + 1):
                if w == 0 or i + w < 0 or i + w >= len(words):
                    continue

                word_lists.append((word, words[i + w]))
    return word_lists, all_text

In [8]:
word_lists, all_text = create_context_pairs(sentences)

In [9]:
import torch


# Creating Unique Word Dictionary
def create_unique_word_dict(text):
    words = list(set(text))
    words.sort()
    return {word: i for i, word in enumerate(words)}


# One-Hot Encoding
def make_one_hot(word_idx, vocab_size):
    x = torch.zeros(vocab_size).float()
    x[word_idx] = 1.0
    return x


unique_word_dict = create_unique_word_dict(all_text)

In [11]:
# Creating one Hot Encoding
make_one_hot(3, len(unique_word_dict))

tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [12]:
from torch import nn, optim


class EnhancedNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(EnhancedNN, self).__init__()
        self.linear1 = nn.Linear(vocab_size, embedding_dim * 2)
        self.linear2 = nn.Linear(embedding_dim * 2, embedding_dim * 2)
        self.linear3 = nn.Linear(embedding_dim * 2, embedding_dim)
        self.linear4 = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        x = torch.relu(self.linear1(x))
        x = torch.relu(self.linear2(x))
        x = torch.relu(self.linear3(x))
        out = self.linear4(x)
        return out


# Model Initialization
vocab_size = len(unique_word_dict)
embedding_dim = 2  # For visualization

#model = SimpleNN(vocab_size, embedding_dim)
model = EnhancedNN(vocab_size, embedding_dim)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Using Adam optimizer

In [14]:
from torch.utils.data import DataLoader, TensorDataset

# Create dataset for DataLoader
X_train = []
Y_train = []

for context, target in word_lists:
    context_idx = unique_word_dict[context]
    target_idx = unique_word_dict[target]
    X_train.append(make_one_hot(context_idx, vocab_size))
    Y_train.append(torch.tensor(target_idx, dtype=torch.long))

X_train = torch.stack(X_train)
Y_train = torch.stack(Y_train)
train_data = TensorDataset(X_train, Y_train)
train_loader = DataLoader(train_data, shuffle=True, batch_size=256)

# Training Loop
for epoch in range(4000):
    total_loss = 0

    for context, target in train_loader:
        model.zero_grad()
        log_probs = model(context)
        loss = loss_function(log_probs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {total_loss}")

Epoch 0, Loss: 6.503991365432739
Epoch 1, Loss: 6.191897630691528
Epoch 2, Loss: 6.727720499038696
Epoch 3, Loss: 7.079899311065674
Epoch 4, Loss: 6.925006151199341
Epoch 5, Loss: 6.004735708236694
Epoch 6, Loss: 5.755437850952148
Epoch 7, Loss: 6.93203067779541
Epoch 8, Loss: 6.6682939529418945
Epoch 9, Loss: 6.6885364055633545
Epoch 10, Loss: 7.373963356018066
Epoch 11, Loss: 6.808918714523315
Epoch 12, Loss: 6.390919923782349
Epoch 13, Loss: 6.61242151260376
Epoch 14, Loss: 6.491022109985352
Epoch 15, Loss: 7.175556421279907
Epoch 16, Loss: 6.439912796020508
Epoch 17, Loss: 6.223335027694702
Epoch 18, Loss: 6.433818101882935
Epoch 19, Loss: 6.194533109664917
Epoch 20, Loss: 6.4850993156433105
Epoch 21, Loss: 6.240411043167114
Epoch 22, Loss: 6.164805889129639
Epoch 23, Loss: 5.733671188354492
Epoch 24, Loss: 6.279323101043701
Epoch 25, Loss: 6.304452657699585
Epoch 26, Loss: 5.962826490402222
Epoch 27, Loss: 6.764610767364502
Epoch 28, Loss: 6.864917039871216
Epoch 29, Loss: 6.45945

# PyTorch embedding layer

In [15]:
import torch.nn as nn

embedding = nn.Embedding(num_embeddings=10000, embedding_dim=300)

In [16]:
#using Embedding Layers
import torch

# Create an embedding object
embedding = nn.Embedding(10, 3)  # 10 words in vocab, 3 dimensional embeddings

# Indices of words
input = torch.LongTensor([1, 2, 8, 4])

# Get embeddings for input indices
output = embedding(input)
print(output)

tensor([[ 1.0228,  1.1517, -0.5884],
        [ 0.5665,  1.5728,  1.5514],
        [-1.8791,  1.4759, -0.7581],
        [-1.5699,  0.3082,  2.6439]], grad_fn=<EmbeddingBackward0>)


In [17]:
class Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

    # Additional layers here
    def forward(self, x):
        x = self.embedding(x)

        # Pass through additional layers
        return x

# Comparing two sentences using word embeddings in PyTorch

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Example sentences
sentence1 = "The quick brown fox"
sentence2 = "The lazy dog"

# Tokenize sentences (naively by splitting on spaces, in practice use a proper tokenizer)
tokens1 = sentence1.lower().split()
tokens2 = sentence2.lower().split()

# Assume a vocabulary mapping tokens to indices
vocab = {"the": 0, "quick": 1, "brown": 2, "fox": 3, "lazy": 4, "dog": 5}
vocab_size = len(vocab)

# Map tokens to indices
indices1 = [vocab[token] for token in tokens1 if token in vocab]
indices2 = [vocab[token] for token in tokens2 if token in vocab]

# Embedding layer
embedding_dim = 50
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

# Convert indices to tensors
indices_tensor1 = torch.tensor(indices1, dtype=torch.long)
indices_tensor2 = torch.tensor(indices2, dtype=torch.long)

# Get embeddings
embeddings1 = embedding(indices_tensor1)
embeddings2 = embedding(indices_tensor2)

# Aggregate embeddings (e.g., by averaging)
sentence_embedding1 = torch.mean(embeddings1, dim=0)
sentence_embedding2 = torch.mean(embeddings2, dim=0)

# Compare embeddings (e.g., using cosine similarity)
cosine_sim = F.cosine_similarity(sentence_embedding1.unsqueeze(0), sentence_embedding2.unsqueeze(0))
print("Cosine Similarity:", cosine_sim.item())

Cosine Similarity: 0.3383994400501251


# Implementing stacked LSTM in PyTorch

In [20]:
import torch
import torch.nn as nn


class StackedLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(StackedLSTM, self).__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out


# Example usage
input_size = 10
hidden_size = 50
num_layers = 2
output_size = 1
model = StackedLSTM(input_size, hidden_size, num_layers, output_size)
print(model)

StackedLSTM(
  (lstm): LSTM(10, 50, num_layers=2, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)


In [23]:
# 361